In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import time

def show(img, title=""):
    plt.figure(figsize=(12, 5))
    plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


## 1. Load và tiền xử lý ảnh

In [ ]:
# Thứ tự: hinh4 → hinh1 (từ phải sang trái theo chiều chụp)
# Pipeline ghép từ giữa ra hai phía, lấy hinh3 làm ảnh tham chiếu gốc
paths = [
    '20240409_095613.jpg',  # hinh4
    '20240409_095607.jpg',  # hinh3
    '20240409_095602.jpg',  # hinh2
    '20240409_095557.jpg',  # hinh1
]

images = []
for path in paths:
    img = cv.imread(path)
    if img is None:
        raise FileNotFoundError(f'Không tìm thấy ảnh: {path}')
    # Chuẩn hóa kích thước để đảm bảo tính nhất quán giữa các ảnh
    img = cv.resize(img, (800, 400))
    images.append(img)
    print(f'Loaded {path} -> {img.shape}')

show(cv.hconcat(images), 'Input: 4 ảnh đầu vào (hinh4 → hinh1)')


## 2. Pipeline ghép từng cặp ảnh

In [ ]:
def stitch_pair_pro(img_left, img_right, method='SIFT'):
    """Ghép img_right vào không gian tọa độ của img_left."""

    # Chuyển sang ảnh xám — detector chỉ cần thông tin cường độ
    gray_l = cv.cvtColor(img_left,  cv.COLOR_BGR2GRAY)
    gray_r = cv.cvtColor(img_right, cv.COLOR_BGR2GRAY)

    # ── Trích xuất đặc trưng ──────────────────────────────────────────────────
    if method == 'SIFT':
        finder = cv.SIFT_create()
        kp1, des1 = finder.detectAndCompute(gray_l, None)
        kp2, des2 = finder.detectAndCompute(gray_r, None)

        # FLANN (KD-tree) — tìm kiếm láng giềng xấp xỉ nhanh cho descriptor float
        matcher = cv.FlannBasedMatcher(
            dict(algorithm=1, trees=5),  # FLANN_INDEX_KDTREE
            dict(checks=50)
        )
        matches = matcher.knnMatch(des1, des2, k=2)
        # Lowe's ratio test — loại bỏ các match mơ hồ
        good_matches = [m for m, n in matches if m.distance < 0.7 * n.distance]

    else:  # ORB
        finder = cv.ORB_create(nfeatures=3000)
        kp1, des1 = finder.detectAndCompute(gray_l, None)
        kp2, des2 = finder.detectAndCompute(gray_r, None)

        # BFMatcher với Hamming distance — phù hợp descriptor nhị phân của ORB
        matcher = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=True)
        good_matches = sorted(matcher.match(des1, des2),
                               key=lambda x: x.distance)[:100]

    print(f'  [{method}] keypoints: {len(kp1)} | {len(kp2)} '
          f'— good matches: {len(good_matches)}')

    if len(good_matches) < 10:
        print('  Cảnh báo: không đủ matches, bỏ qua bước ghép này.')
        return img_left

    # ── Ước lượng Homography ──────────────────────────────────────────────────
    # DLT + RANSAC: tính H ánh xạ img_right về hệ tọa độ img_left
    pts_l = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    pts_r = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    H, _ = cv.findHomography(pts_r, pts_l, cv.RANSAC, 5.0)

    if H is None:
        print('  Cảnh báo: không ước lượng được H, bỏ qua bước ghép này.')
        return img_left

    # ── Warping ───────────────────────────────────────────────────────────────
    # Mở rộng canvas để chứa cả hai ảnh sau khi căn chỉnh
    h_l, w_l = img_left.shape[:2]
    result = cv.warpPerspective(img_right, H, (w_l * 2, h_l))

    # ── Feather blending ──────────────────────────────────────────────────────
    # Mask trọng số mờ dần tại vùng chồng lấn — tránh đường viền cứng
    mask_l = np.zeros((h_l, w_l * 2), dtype=np.float32)
    mask_l[:, :w_l] = 1.0
    mask_l = cv.GaussianBlur(mask_l, (51, 51), 0)

    canvas_left = np.zeros_like(result)
    canvas_left[:, :w_l] = img_left

    blended = (canvas_left * mask_l[:, :, np.newaxis]
               + result    * (1.0 - mask_l[:, :, np.newaxis])).astype(np.uint8)

    # ── Crop vùng đen thừa ────────────────────────────────────────────────────
    gray_out = cv.cvtColor(blended, cv.COLOR_BGR2GRAY)
    cnts, _ = cv.findContours((gray_out > 0).astype(np.uint8),
                               cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        x, y, w, h = cv.boundingRect(cnts[0])
        blended = blended[y:y+h, x:x+w]

    return blended


## 3. Ghép 4 ảnh thành panorama

In [ ]:
def run_full_pipeline(img_list, method='SIFT'):
    """
    Ghép 4 ảnh theo chiến lược từ giữa ra hai phía.
    img_list = [hinh4, hinh3, hinh2, hinh1]
    """
    print(f'--- Phương pháp: {method} ---')
    t0 = time.time()

    # Bước 1 & 2: ghép song song hai phía, lấy hinh3 làm tham chiếu trung tâm
    print('Bước 1: hinh3 + hinh4 → left_part')
    left_part  = stitch_pair_pro(img_list[1], img_list[0], method)

    print('Bước 2: hinh2 + hinh1 → right_part')
    right_part = stitch_pair_pro(img_list[2], img_list[3], method)

    # Bước 3: hợp nhất hai khối thành panorama hoàn chỉnh
    print('Bước 3: left_part + right_part → panorama')
    panorama   = stitch_pair_pro(left_part, right_part, method)

    print(f'Hoàn thành trong {time.time() - t0:.2f}s\n')
    return panorama


panorama_sift = run_full_pipeline(images, method='SIFT')
panorama_orb  = run_full_pipeline(images, method='ORB')

show(panorama_sift, 'Panorama — SIFT')
show(panorama_orb,  'Panorama — ORB')


## 4. So sánh SIFT vs ORB

In [ ]:
# Resize về cùng chiều cao để hiển thị cạnh nhau
h_target = min(panorama_sift.shape[0], panorama_orb.shape[0])

def resize_h(img, h):
    scale = h / img.shape[0]
    return cv.resize(img, (int(img.shape[1] * scale), h))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].imshow(cv.cvtColor(resize_h(panorama_sift, h_target), cv.COLOR_BGR2RGB))
axes[0].set_title('SIFT', fontsize=13)
axes[0].axis('off')
axes[1].imshow(cv.cvtColor(resize_h(panorama_orb, h_target), cv.COLOR_BGR2RGB))
axes[1].set_title('ORB', fontsize=13)
axes[1].axis('off')
plt.suptitle('So sánh SIFT vs ORB', fontsize=14)
plt.tight_layout()
plt.show()


## 5. Tham khảo: OpenCV Stitcher (tự động)

Sử dụng `cv.Stitcher` để đối chiếu — pipeline hoàn toàn tự động của OpenCV.

In [ ]:
# cv.Stitcher xử lý toàn bộ pipeline nội bộ (feature, match, warp, blend)
# Dùng để đối chiếu trực quan với pipeline thủ công ở trên
stitcher = cv.Stitcher.create()
status, pano_auto = stitcher.stitch(images)

if status == cv.STITCHER_OK:
    show(pano_auto, 'Panorama — OpenCV Stitcher (tự động)')
else:
    print(f'Stitcher thất bại, status code: {status}')
